![image_1780161611277.png](./image_1780161611277.png "image_1780161611277.png")

![image_1780161651148.png](./image_1780161651148.png "image_1780161651148.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
# Initialize Spark
spark = SparkSession.builder.appName("OrdersData").getOrCreate()

# Create DataFrame
orders_data = [
    (1, 101, "completed"),
    (2, 101, "returned"),
    (3, 101, "returned"),
    (4, 102, "completed"),
    (5, 102, "completed"),
    (6, 102, "completed"),
    (7, 103, "returned"),
    (8, 103, "completed"),
    (9, 103, "returned"),
    (10, 103, "returned"),
    (11, 104, "completed"),
    (12, 104, "completed"),
]

orders_df = spark.createDataFrame(orders_data, ["id", "product_id", "status"])

# Show DataFrame
orders_df.show()


In [0]:
result_df = (
    orders_df.groupBy("product_id")
    .agg(
        f.count("status").alias("total_orders"),
        f.sum(f.when(f.col("status") == "returned", 1).otherwise(0)).alias(
            "returned_orders"
        ),
    )
    .select(
        f.col("product_id"),
        f.col("total_orders"),
        f.col("returned_orders"),
        f.round(f.col("returned_orders") / f.col("total_orders"), 2).alias(
            "return_rate"
        ),
    )
    .orderBy(f.col("return_rate").desc(), f.col("product_id").asc())
)
display(result_df)